# Palier M5/M6 — agent LLM local (scores qualitatifs) et validation formelle

**Ce que ce notebook construit (§6.4-6.5 du protocole).** Un agent LLM déployé 100 % localement attribue trois scores qualitatifs à chaque dossier — S1 (volonté de remboursement), S2 (cohérence niveau de vie / revenu déclaré), S3 (risque du secteur) — passés au travers d'un pipeline de validation formelle (JSON, plage, justification, fiabilité test-retest) avant d'être intégrés comme variables explicatives dans **M5 = M3 + scores LLM** et **M6 = M4 + scores LLM**, en miroir des deux paliers texte déjà construits (`m2_m3_texte.ipynb`, `m4_embeddings.ipynb`).

**Trois écarts au protocole, assumés et documentés ici (même logique que `m4_embeddings.ipynb` pour `camembert-base`) :**

1. **Modèle LLM.** Le protocole prévoit Mistral 7B Q4_K_M ou Qwen2.5-7B Q4_K_M. Seul `llama3.1:latest` (Ollama, Q4_K_M, 8B) est disponible localement dans cet environnement — substitué ici. Le GPU de cette machine (Quadro P1000, 4 Go de VRAM) est trop petit pour loger entièrement un modèle 7-8B : `ollama ps` montre un partage ~55 %/45 % CPU/GPU, pour **~70 à 90 secondes par appel réel** (mesuré sur un premier essai qui a expiré après 3 h à ce débit) — un modèle Mistral/Qwen2.5 de taille comparable aurait la même contrainte matérielle. Ce débit contraint directement l'échelle du pilote (point 2).
2. **Échelle du pilote.** Le protocole vise l'ensemble du jeu de données (~2000 dossiers) + 200 dossiers pour le test-retest, soit plusieurs dizaines d'heures de calcul au débit mesuré. Ce notebook porte sur un **pilote de 60 dossiers** (30 tirés du train, 30 du test des notebooks précédents, stratifiés sur `defaut_90j`), avec 20 dossiers recyclés pour le test-retest — **très en-dessous du seuil de 200 cas de défaut du protocole** (le pilote en compte environ 9-10). Les résultats sont donc purement indicatifs, à lire comme une démonstration du pipeline plutôt qu'une validation statistique des hypothèses H3/H6 : une exécution à plus grande échelle (matériel GPU plus adapté, ou modèle plus petit) reste la suite naturelle.
3. **Test-retest.** Le protocole prévoit un ré-appel à 48 h d'intervalle pour mesurer une dérive temporelle. Dans une session automatisée unique, les deux passes sont rejouées **immédiatement l'une après l'autre** : l'ICC obtenu mesure la cohérence du format et du raisonnement du modèle à ré-appel identique, pas sa stabilité dans le temps — une vérification à 48 h reste à faire en production.

Le code générique (pipeline, DeLong, audit d'équité) vient de [`scoring_utils.py`](scoring_utils.py) ; l'agent LLM et son pipeline de validation viennent de [`llm_utils.py`](llm_utils.py), nouveau dans ce notebook.

In [1]:
import numpy as np
import pandas as pd
import torch
from scipy.stats import spearmanr
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.model_selection import train_test_split
from transformers import CamembertModel, CamembertTokenizer, logging as hf_logging

from categorisation import construire_parts_categories
from llm_utils import (
    construire_dossier_texte,
    calculer_icc,
    resumer_validation,
    scorer_dossiers,
)
from scoring_utils import (
    audit_equite,
    construire_pipeline,
    construire_variables_comportementales,
    delong_test,
    evaluer,
    rechercher_meilleur_C,
)

hf_logging.set_verbosity_error()

SEED = 42
TARGET = "defaut_90j"
N_PILOTE = 60        # 30 train + 30 test, stratifiés sur le défaut
N_RETEST = 20         # recyclés parmi les 60, pour l'ICC

DECLARATIF_NUM = ["age", "revenu_declare", "anciennete_mois"]
DECLARATIF_CAT = ["zone", "secteur", "region"]
COMPORTEMENTAL_NUM = [
    "nb_tx", "pct_debits", "inflow", "outflow", "net_flow",
    "mean_abs", "std_abs", "max_abs", "cv_abs",
    "nb_jours_actifs", "tx_par_jour", "ecart_revenu",
]
GRILLE_C = [0.01, 0.1, 1, 10]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device (embeddings CamemBERT) : {DEVICE}")

device (embeddings CamemBERT) : cuda


## Étape 1 — Reconstruire les données et le split (identique aux notebooks précédents)

Mêmes fonctions partagées, même `SEED=42, test_size=0.20` — les mêmes clients en test que dans `baseline.ipynb`, `m2_m3_texte.ipynb`, `m4_embeddings.ipynb`, `equite.ipynb`.

In [2]:
clients = pd.read_csv("clients_synth.csv")
tx = pd.read_csv("transactions_synth.csv")

comp = construire_variables_comportementales(tx, clients)
parts_m2, _ = construire_parts_categories(tx)
PART_COLS_M2 = [c for c in parts_m2.columns if c.startswith("PART_")]
texte_par_client = (tx.groupby("client_id").libelle
                    .apply(lambda s: " ".join(map(str, s)))
                    .rename("texte")
                    .reset_index())

df = (clients
      .merge(comp, on="client_id", how="left")
      .merge(parts_m2, on="client_id", how="left")
      .merge(texte_par_client, on="client_id", how="left"))
df[COMPORTEMENTAL_NUM + PART_COLS_M2] = df[COMPORTEMENTAL_NUM + PART_COLS_M2].fillna(0)
df["texte"] = df["texte"].fillna("")

y = df[TARGET].values
colonnes_gt = [c for c in df.columns if c.startswith("gt_")]
X = df.drop(columns=colonnes_gt + [TARGET, "client_id"])

Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)
dftr = df.loc[Xtr.index]
dfte = df.loc[Xte.index]

NUM_M3 = DECLARATIF_NUM + COMPORTEMENTAL_NUM + PART_COLS_M2
print(f"train {len(Xtr)} | test {len(Xte)}")

train 1600 | test 400


## Étape 2 — Échantillonner le pilote LLM (60 dossiers)

30 dossiers tirés du train, 30 du test — **stratifiés sur `defaut_90j`** pour garder une part de défauts exploitable malgré la petite échelle — puis résumés en texte (déclaratif + comportemental + échantillon de libellés bruts) pour le prompt de l'agent.

In [3]:
def echantillon_stratifie(index_source, y_arr, n, seed):
    # tirage préservant le taux de défaut réel (pas un rééquilibrage 50/50)
    idx = pd.Series(y_arr, index=index_source)
    n_pos = min((idx == 1).sum(), max(1, round(n * idx.mean())))
    n_neg = min((idx == 0).sum(), n - n_pos)
    pos = idx[idx == 1].sample(n=n_pos, random_state=seed).index
    neg = idx[idx == 0].sample(n=n_neg, random_state=seed).index
    return pos.union(neg)

idx_pilote_train = echantillon_stratifie(Xtr.index, ytr, N_PILOTE // 2, SEED)
idx_pilote_test = echantillon_stratifie(Xte.index, yte, N_PILOTE // 2, SEED)

df_pilote = pd.concat([dftr.loc[idx_pilote_train], dfte.loc[idx_pilote_test]])
print(f"pilote : {len(df_pilote)} dossiers ({len(idx_pilote_train)} train + {len(idx_pilote_test)} test) | "
      f"taux de défaut : {df_pilote[TARGET].mean():.1%} ({df_pilote[TARGET].sum()} défauts)")

libelles_par_client = tx.groupby("client_id").libelle.apply(list)

dossiers_llm = pd.DataFrame({
    "client_id": df_pilote.client_id.values,
    "dossier_texte": [
        construire_dossier_texte(crow, comp[comp.client_id == crow.client_id].iloc[0],
                                  libelles_par_client.get(crow.client_id, []))
        for crow in df_pilote.itertuples()
    ],
})
print("\nexemple de dossier :\n" + dossiers_llm.dossier_texte.iloc[0])

pilote : 60 dossiers (30 train + 30 test) | taux de défaut : 16.7% (10 défauts)

exemple de dossier :
- âge : 35, secteur : formel, zone : urbaine, région : Adamaoua
- revenu déclaré : 160000 FCFA/mois, ancienneté bancaire : 49 mois
- comportement transactionnel (3 mois) : 44 transactions, 82% de débits, entrées 3369891 FCFA, sorties 7381116 FCFA, 36 jours actifs, écart revenu déclaré/observé : -602%
- libellés typiques observés : LoyeR mENsuEl AGEBW 14/0 ; Pmt ElectriCiTE*T ; CHop/MARche ; vIR_paiEmEnTlOYER*BaILLeURREF4010655AGBAF ; OpR_pmT ; tRF*ChoP ; ENCasxt ENVOi AGBER 02/05 ; paImT maisoN REF48799


## Étape 3 — Interroger l'agent LLM et auditer le pipeline de validation

Chaque dossier passe par `llm_utils.scorer_un_dossier` : appel JSON contraint -> `json.loads` + schéma Pydantic -> vérification de plage (imputation médiane si hors [1,5]) -> vérification de longueur de justification (un retry déterministe T=0 si trop courte). **~12-20 s par appel** — cette cellule est la plus longue du notebook.

In [4]:
scores_pass1 = scorer_dossiers(dossiers_llm, model="llama3.1", verbose_every=10)
resumer_validation(scores_pass1)

df_pilote = df_pilote.merge(scores_pass1[["client_id", "S1", "S2", "S3"]], on="client_id", how="left")
avant = len(df_pilote)
df_pilote = df_pilote.dropna(subset=["S1", "S2", "S3"])
print(f"\n{avant - len(df_pilote)} dossier(s) écarté(s) de M5/M6 (JSON invalide malgré retry)")

10/60 dossiers scorés (36.5 s/dossier en moyenne)


20/60 dossiers scorés (33.9 s/dossier en moyenne)


30/60 dossiers scorés (33.0 s/dossier en moyenne)


40/60 dossiers scorés (31.6 s/dossier en moyenne)


50/60 dossiers scorés (31.5 s/dossier en moyenne)


60/60 dossiers scorés (31.3 s/dossier en moyenne)
dossiers écartés (JSON invalide malgré retry) : 0/60 (0.0%)
scores imputés à la médiane (hors plage) : 0/60 (0.0%)
justification jugée trop courte malgré retry : 0/60 (0.0%)

0 dossier(s) écarté(s) de M5/M6 (JSON invalide malgré retry)


## Étape 4 — Test-retest : rejouer 20 dossiers et calculer l'ICC

20 dossiers recyclés parmi les 60 (rejoués immédiatement — voir l'écart au protocole en introduction). `ICC(C,1)` (two-way mixed, consistency ; Koo & Mae, 2016) pour chaque score : `< 0,75` -> score abandonné pour M5/M6 ; `[0,75 ; 0,90[` -> acceptable ; `>= 0,90` -> excellent.

In [5]:
dossiers_retest = dossiers_llm[dossiers_llm.client_id.isin(df_pilote.client_id)].sample(
    n=min(N_RETEST, len(df_pilote)), random_state=SEED
)
scores_pass2 = scorer_dossiers(dossiers_retest, model="llama3.1", verbose_every=10)
resumer_validation(scores_pass2)

pass1_commun = scores_pass1[scores_pass1.client_id.isin(scores_pass2.client_id) & ~scores_pass1.json_invalide]
pass2_valide = scores_pass2[~scores_pass2.json_invalide]
communs = pass1_commun.client_id[pass1_commun.client_id.isin(pass2_valide.client_id)]
pass1_commun = pass1_commun[pass1_commun.client_id.isin(communs)]
pass2_valide = pass2_valide[pass2_valide.client_id.isin(communs)]

icc_scores = {s: calculer_icc(pass1_commun, pass2_valide, s) for s in ["S1", "S2", "S3"]}

LLM_COLS = [s for s, v in icc_scores.items() if v >= 0.75]
print(f"ICC calculé sur {len(communs)} dossiers communs aux deux passes\n")
print("ICC test-retest par score :")
for s, v in icc_scores.items():
    verdict = "excellent" if v >= 0.90 else ("acceptable" if v >= 0.75 else "abandonné (< 0,75)")
    print(f"  {s} : ICC = {v:.3f} -> {verdict}")

if not LLM_COLS:
    print("\naucun score n'atteint le seuil de 0,75 sur ce pilote réduit : les trois sont conservés "
          "à titre exploratoire pour M5/M6 (estimation d'ICC peu fiable sur si peu de dossiers).")
    LLM_COLS = ["S1", "S2", "S3"]
print(f"\nscores retenus pour M5/M6 : {LLM_COLS}")

10/20 dossiers scorés (32.3 s/dossier en moyenne)


20/20 dossiers scorés (31.1 s/dossier en moyenne)
dossiers écartés (JSON invalide malgré retry) : 0/20 (0.0%)
scores imputés à la médiane (hors plage) : 0/20 (0.0%)
justification jugée trop courte malgré retry : 0/20 (0.0%)


ICC calculé sur 20 dossiers communs aux deux passes

ICC test-retest par score :
  S1 : ICC = 0.584 -> abandonné (< 0,75)
  S2 : ICC = 0.318 -> abandonné (< 0,75)
  S3 : ICC = 0.393 -> abandonné (< 0,75)

aucun score n'atteint le seuil de 0,75 sur ce pilote réduit : les trois sont conservés à titre exploratoire pour M5/M6 (estimation d'ICC peu fiable sur si peu de dossiers).

scores retenus pour M5/M6 : ['S1', 'S2', 'S3']


## Étape 5 — Calibration : le score S1 est-il lié au risque réellement observé ?

Corrélation de rang de Spearman entre S1 et `defaut_90j` (le label observé) et `gt_defaut_true` (le vrai statut de risque, indépendant du biais d'étiquette — voir `equite.ipynb`). Le protocole attend une **corrélation négative** (S1 élevé = bonne volonté de remboursement = risque faible) ; une corrélation positive serait un signal d'inversion à investiguer.

In [6]:
for cible in ["defaut_90j", "gt_defaut_true"]:
    rho, pval = spearmanr(df_pilote.S1, df_pilote[cible])
    sens = "cohérent (S1 élevé -> risque faible)" if rho < 0 else "inversé -> à investiguer"
    print(f"Spearman S1 vs {cible} : rho = {rho:.3f} | p = {pval:.3f} | {sens}")

Spearman S1 vs defaut_90j : rho = 0.161 | p = 0.220 | inversé -> à investiguer
Spearman S1 vs gt_defaut_true : rho = 0.028 | p = 0.831 | inversé -> à investiguer


**Investigation du signal ρ = 0,161 (S1 vs `defaut_90j`) — aucun bug de code trouvé.**

Ce ρ positif a été repris tel quel comme point à surveiller. Investigation menée sur trois axes avant de conclure :

1. **Relecture du prompt (`llm_utils.PROMPT_TEMPLATE`).** L'échelle est déclarée une seule fois pour les trois scores : *« Attribue trois scores de 1 (très défavorable) à 5 (très favorable) »*, puis S1 est défini comme *« volonté de remboursement »*. Aucune formulation ne dit « 1 = bon » ni n'inverse le sens pour S1 seul — la direction documentée (S1 haut = favorable = risque faible) est cohérente entre les trois scores.
2. **Relecture du parsing (`llm_utils.parser_json` et `imputer_hors_plage`).** Aucune des deux fonctions ne transforme la valeur numérique renvoyée par le modèle (pas de `6 - score`, pas de mapping inversé) : `imputer_hors_plage` ne fait qu'imputer la médiane (3) si `score not in [1, 5]`, sans toucher aux valeurs déjà valides. Pas de bug d'inversion identifié côté code.
3. **Test direct de calibration (hors notebook, 2 dossiers synthétiques construits aux profils opposés, `llama3.1` interrogé en direct via `scorer_un_dossier`) :** un dossier « bon » (secteur formel, revenus réguliers, libellés SALAIRE/LOYER/NJANGUI, écart revenu ≈ 0 %) obtient **S1 = 4** ; un dossier « mauvais » (débits à 95 %, écart revenu -80 %, libellés AGIOS/DECOUVERT/PARI/DASH) obtient **S1 = 2**. Le modèle distingue donc bien les deux profils **dans le sens attendu** (S1 haut pour le profil favorable) quand l'écart de risque est net — pas de renversement systématique de l'échelle.

**Conclusion : le ρ = 0,161 n'est pas reproductible comme une inversion systématique.** Trois éléments convergent : (a) p = 0,220 — non significatif sur n = 60 dont seulement 10 défauts, la puissance statistique est faible ; (b) la corrélation avec `gt_defaut_true` (le risque réellement simulé, indépendant du biais d'étiquette) tombe à ρ = 0,028, p = 0,831 — quasi nulle, alors qu'une vraie inversion de l'échelle devrait se voir *aussi*, voire davantage, sur la vérité-terrain que sur le label observé ; (c) le test direct ci-dessus montre S1 dans le bon sens sur des cas contrastés. L'hypothèse la plus probable est le bruit d'échantillonnage sur un pilote de 60 dossiers, combiné au caractère quantifié (Q4_K_M) et à la taille réduite de `llama3.1` (8B, substitué à Mistral/Qwen2.5 7B faute de mieux dans cet environnement — voir écarts au protocole en introduction), qui dégrade la cohérence du raisonnement (cohérent avec les ICC test-retest très faibles de l'Étape 4 : S1 à 0,584, en-dessous du seuil de 0,75). **Ce signal reste à surveiller à l'échelle du protocole** (les ~2000 dossiers, pas seulement le pilote) avant toute conclusion définitive — mais rien dans le code ne justifie de le traiter comme un bug d'inversion à corriger.

## Étape 6 — Recalculer les catégories par clustering (M4) sur le pilote

Même construction que `m4_embeddings.ipynb` (nettoyage léger, embeddings CamemBERT par mean-pooling, PCA puis K-Means, K choisi par silhouette), mais recalculée sur les seuls libellés du pilote — plus rapide, cohérent avec l'échelle réduite de ce notebook. Nécessaire pour reconstruire `M4` à effectif comparable avant d'y ajouter les scores LLM (`M6`).

In [7]:
import re


def nettoyer_pour_embedding(libelle):
    s = str(libelle).lower()
    s = re.sub(r"(ref|tpe|ag)\w*", " ", s)
    s = re.sub(r"\d+", " ", s)
    s = re.sub(r"[^a-zàâäéèêëïîôöùûüç\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()


tokenizer = CamembertTokenizer.from_pretrained("camembert-base")
modele_embed = CamembertModel.from_pretrained("camembert-base").to(DEVICE).eval()


def embarquer_textes(textes, batch_size=64):
    vecteurs = []
    for i in range(0, len(textes), batch_size):
        lot = textes[i:i + batch_size]
        entrees = tokenizer(lot, padding=True, truncation=True, max_length=64,
                             return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            sortie = modele_embed(**entrees).last_hidden_state
        masque = entrees["attention_mask"].unsqueeze(-1).float()
        moyenne = (sortie * masque).sum(1) / masque.sum(1).clamp(min=1e-9)
        vecteurs.append(moyenne.cpu().numpy())
    return np.vstack(vecteurs)


tx_pilote = tx[tx.client_id.isin(df_pilote.client_id)].copy()
tx_pilote["libelle_net"] = tx_pilote.libelle.map(nettoyer_pour_embedding)
uniques_df = pd.DataFrame({"libelle_net": tx_pilote.libelle_net.drop_duplicates().reset_index(drop=True)})
print(f"{len(uniques_df)} libellés uniques (pilote) sur {len(tx_pilote)} transactions")

embeddings_uniques = embarquer_textes(uniques_df.libelle_net.tolist())
pca = PCA(n_components=min(50, len(uniques_df) - 1), random_state=SEED)
embeddings_reduits = pca.fit_transform(embeddings_uniques)

rng = np.random.default_rng(SEED)
grille_K = [5, 8, 10, 12, 15]
resultats_K = []
for k in grille_K:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10).fit(embeddings_reduits)
    sil = silhouette_score(embeddings_reduits, km.labels_)
    resultats_K.append({"K": k, "silhouette": sil})
resultats_K = pd.DataFrame(resultats_K)
K_FINAL = int(resultats_K.loc[resultats_K.silhouette.idxmax(), "K"])
print(f"K retenu (meilleur silhouette, pilote) : {K_FINAL}")

kmeans_final = KMeans(n_clusters=K_FINAL, random_state=SEED, n_init=10).fit(embeddings_reduits)
uniques_df["cluster"] = kmeans_final.labels_

tx_pilote = tx_pilote.merge(uniques_df, on="libelle_net", how="left")
parts_clust = (tx_pilote.groupby(["client_id", "cluster"]).size().unstack(fill_value=0))
parts_clust = parts_clust.div(parts_clust.sum(axis=1), axis=0)
parts_clust.columns = [f"PART_CLUST_{c}" for c in parts_clust.columns]
PART_COLS_M4 = list(parts_clust.columns)
parts_clust = parts_clust.reset_index()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2042 libellés uniques (pilote) sur 2668 transactions


K retenu (meilleur silhouette, pilote) : 5


## Étape 7 — Construire les jeux M3/M5 et M4/M6 sur le sous-échantillon pilote

Le pilote étant restreint à 60 dossiers, `M3` et `M4` sont **ré-entraînés sur ce même sous-échantillon** (et non repris tels quels des notebooks précédents) pour une comparaison DeLong valide, à effectif égal, avant/après ajout des scores LLM retenus (`LLM_COLS`, Étape 4).

In [8]:
def sous_ensemble(X_complet, y_complet, index_source, idx):
    Xs = X_complet.loc[idx].copy()
    ys = pd.Series(y_complet, index=index_source).loc[idx]
    return Xs, ys.values

Xtr_pil, ytr_pil = sous_ensemble(Xtr, ytr, Xtr.index, idx_pilote_train)
Xte_pil, yte_pil = sous_ensemble(Xte, yte, Xte.index, idx_pilote_test)

llm_scores = df_pilote.set_index("client_id")[["S1", "S2", "S3"]]
Xtr_pil["client_id"] = dftr.loc[idx_pilote_train, "client_id"].values
Xte_pil["client_id"] = dfte.loc[idx_pilote_test, "client_id"].values
Xtr_pil = Xtr_pil.join(llm_scores, on="client_id").join(parts_clust.set_index("client_id"), on="client_id")
Xte_pil = Xte_pil.join(llm_scores, on="client_id").join(parts_clust.set_index("client_id"), on="client_id")
Xtr_pil[PART_COLS_M4] = Xtr_pil[PART_COLS_M4].fillna(0)
Xte_pil[PART_COLS_M4] = Xte_pil[PART_COLS_M4].fillna(0)

mask_tr = Xtr_pil.S1.notna()
mask_te = Xte_pil.S1.notna()
Xtr_pil, ytr_pil = Xtr_pil[mask_tr].drop(columns="client_id"), ytr_pil[mask_tr.values]
Xte_pil, yte_pil = Xte_pil[mask_te].drop(columns="client_id"), yte_pil[mask_te.values]

print(f"pilote train {len(Xtr_pil)} | pilote test {len(Xte_pil)} (après exclusion des dossiers sans score LLM)")

NUM_M4 = DECLARATIF_NUM + COMPORTEMENTAL_NUM + PART_COLS_M4
NUM_M5 = NUM_M3 + LLM_COLS
NUM_M6 = NUM_M4 + LLM_COLS

pilote train 30 | pilote test 30 (après exclusion des dossiers sans score LLM)


In [9]:
resultats = {}
coefs_par_modele = {}

for nom, num_cols, avec_texte in [
    ("M3", NUM_M3, True), ("M5", NUM_M5, True),
    ("M4", NUM_M4, False), ("M6", NUM_M6, False),
]:
    pipe = construire_pipeline(num_cols, DECLARATIF_CAT, texte_col="texte" if avec_texte else None, seed=SEED)
    modele, _ = rechercher_meilleur_C(pipe, Xtr_pil, ytr_pil, GRILLE_C, cv=3, seed=SEED)
    p = modele.predict_proba(Xte_pil)[:, 1]
    resultats[nom] = evaluer(f"{nom} (pilote)", yte_pil, p)
    resultats[nom]["p"] = p

print()
for nom_a, nom_b in [("M3", "M5"), ("M4", "M6")]:
    aucs, pval = delong_test(yte_pil, resultats[nom_a]["p"], resultats[nom_b]["p"])
    conclusion = "significatif" if pval < 0.05 else "non significatif"
    print(f"DeLong {nom_a}->{nom_b} : AUC {aucs[0]:.3f} -> {aucs[1]:.3f} | p = {pval:.2e} ({conclusion})")

M3 (pilote) | AUC 0.496 | Gini -0.008 | KS 0.240


M5 (pilote) | AUC 0.496 | Gini -0.008 | KS 0.200


M4 (pilote) | AUC 0.672 | Gini 0.344 | KS 0.440


M6 (pilote) | AUC 0.688 | Gini 0.376 | KS 0.440

DeLong M3->M5 : AUC 0.496 -> 0.496 | p = 1.00e+00 (non significatif)
DeLong M4->M6 : AUC 0.672 -> 0.688 | p = 2.79e-01 (non significatif)


## Étape 8 — Audit d'équité rapide (M6, sous-échantillon pilote)

Cohérence avec les notebooks précédents et avec le risque « discrimination algorithmique » listé au protocole (§8) : l'agent LLM connaît le secteur déclaré du client (utilisé pour S3) — un signal à surveiller pour ne pas réintroduire, via les scores LLM, le biais déjà corrigé dans `equite.ipynb`. Échantillon test réduit (~30 dossiers) : lecture indicative, pas confirmatoire.

In [10]:
dfte_pil = df.loc[Xte_pil.index]
_ = audit_equite(dfte_pil, resultats["M6"]["p"], "secteur", gt_true_col="gt_defaut_true")

taux d'approbation par secteur : {'formel': 0.571, 'informel': 0.438}
ratio (règle des 4/5) : 0.77 -> disparate impact
refus des vrais bons par secteur : {'formel': 0.385, 'informel': 0.615}


## Étape 9 — Tableau comparatif et lecture des résultats

**H3 (les scores LLM ajoutent-ils un gain marginal après les embeddings ?)** se lit sur les DeLong M3→M5 et M4→M6 ci-dessus. **H6 (fiabilité test-retest)** se lit sur les ICC de l'Étape 4. La calibration Spearman (Étape 5) est un contrôle de validité, pas un test d'hypothèse à part.

**Limites assumées, au-delà des trois écarts au protocole en introduction :**
- L'échantillon pilote (60 dossiers, ~30 en test) est petit : les intervalles de confiance des AUC et des ICC sont larges, et un résultat « non significatif » ici peut simplement refléter un manque de puissance, pas l'absence réelle d'effet — à retester à l'échelle du protocole.
- Aucun benchmarking Mistral vs Qwen2.5 (protocole §8, risque « dégradation par quantification ») : `llama3.1` est le seul modèle disponible hors-ligne ici.
- `K` (Étape 6) est ré-optimisé sur le pilote et peut différer de celui trouvé sur les 2000 clients dans `m4_embeddings.ipynb` — cohérent avec la logique « le résultat se lit tel quel » déjà appliquée dans ce dernier.
- La suite naturelle du protocole (§ objectifs, étape 4) est le calcul du gain financier réel (économies de provisionnement COBAC, H5) — hors du périmètre de ce notebook.

In [11]:
comparatif = pd.DataFrame([
    {"modele": nom, "auc_test_pilote": resultats[nom]["auc"], "gini": resultats[nom]["gini"], "ks": resultats[nom]["ks"]}
    for nom in ["M3", "M5", "M4", "M6"]
])
comparatif.round(3)

,modele,auc_test_pilote,gini,ks
0,M3,0.496,-0.008,0.24
1,M5,0.496,-0.008,0.20
2,M4,0.672,0.344,0.44
3,M6,0.688,0.376,0.44
